Import Library, Membuat SparkSession, dan Koneksi MongoDB
-

In [1]:
import random
import math
import pandas as pd

from pymongo import MongoClient

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline, PipelineModel

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

# Mengatur random seed
random.seed(1301)

# Membuat koneksi MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["ecommerce_p13"]
koleksi_transaksi = db["transaksi_mentah"]

print("Koneksi MongoDB berhasil.")

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas13-Ecommerce") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat.")
print("Versi Spark:", spark.version)

Koneksi MongoDB berhasil.


26/11/26 14:35:34 WARN Utils: Your hostname, zaydann resolves to a loopback address: 127.0.1.1; using 10.90.67.77 instead (on interface wlo1)
26/11/26 14:35:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 14:35:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat.
Versi Spark: 3.5.9


A. Menyisipkan Data ke MongoDB
-

Membuat Dataset Transaksi Dummy

In [2]:
# Menentukan kategori produk
kategori_list = [
    "Elektronik", "Fashion", "Makanan",
    "Rumah Tangga", "Kesehatan"
]

# Menentukan daftar kota
kota_list = [
    "Magelang", "Yogyakarta", "Semarang", "Solo"
]

# Menentukan metode pembayaran
metode_list = [
    "Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"
]

# Membuat fungsi untuk menghasilkan data transaksi
def buat_transaksi():
    estimasi = random.randint(1, 14)

    # Semakin lama pengiriman, semakin besar peluang pembatalan
    prob_batal = 1 / (1 + math.exp(-(estimasi - 7) / 2))

    return {
        "kategori": random.choice(kategori_list),
        "kota": random.choice(kota_list),
        "unit_terjual": random.randint(1, 15),
        "harga_satuan": random.choice(
            [25000, 50000, 75000, 100000, 150000, 250000]
        ),
        "metode_pembayaran": random.choice(metode_list),
        "estimasi_pengiriman_hari": estimasi,
        "dibatalkan": int(random.random() < prob_batal)
    }

# Membuat 1.000 dokumen transaksi
dokumen_transaksi = [
    buat_transaksi() for _ in range(1000)
]

print("Jumlah data yang dibuat:", len(dokumen_transaksi))

Jumlah data yang dibuat: 1000


Menyimpan dan Memeriksa Data MongoDB

In [3]:
# Menghapus data sebelumnya agar tidak terjadi penumpukan
koleksi_transaksi.delete_many({})

# Menyisipkan data ke MongoDB
hasil_insert = koleksi_transaksi.insert_many(
    dokumen_transaksi
)

print("Jumlah dokumen berhasil disisipkan:", len(hasil_insert.inserted_ids))

# Memeriksa jumlah dokumen dalam koleksi
print(
    "Jumlah dokumen di MongoDB:",
    koleksi_transaksi.count_documents({})
)

# Menampilkan 5 dokumen pertama
sampel = list(
    koleksi_transaksi.find({}, {"_id": 0}).limit(5)
)

pd.DataFrame(sampel)

Jumlah dokumen berhasil disisipkan: 1000
Jumlah dokumen di MongoDB: 1000


,kategori,kota,unit_terjual,harga_satuan,metode_pembayaran,estimasi_pengiriman_hari,dibatalkan
0,Rumah Tangga,Solo,14,75000,Transfer Bank,14,1
1,Makanan,Solo,2,100000,COD,1,0
2,Elektronik,Yogyakarta,7,75000,Kartu Kredit,14,1
3,Elektronik,Solo,10,150000,COD,9,1
4,Makanan,Solo,4,75000,E-Wallet,4,0


B. Ekstraksi ke Spark DataFrame
-

Mengambil Data MongoDB dan Mengubahnya Menjadi Spark DataFrame

In [4]:
# Mengambil seluruh data dari MongoDB
dokumen_mongo = list(
    koleksi_transaksi.find({}, {"_id": 0})
)

# Mengubah data menjadi Pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

print("Jumlah data hasil ekstraksi:", len(df_pandas))

# Mengubah Pandas DataFrame menjadi Spark DataFrame
df = spark.createDataFrame(df_pandas)

print("Data berhasil diubah menjadi Spark DataFrame.")

Jumlah data hasil ekstraksi: 1000
Data berhasil diubah menjadi Spark DataFrame.


Memeriksa Struktur dan Isi Data

In [5]:
# Menampilkan struktur data
df.printSchema()

# Menampilkan jumlah baris
print("Jumlah baris:", df.count())

# Menampilkan 5 data pertama
df.show(5, truncate=False)

root
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: long (nullable = true)
 |-- harga_satuan: long (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- estimasi_pengiriman_hari: long (nullable = true)
 |-- dibatalkan: long (nullable = true)



Jumlah baris: 1000
+------------+----------+------------+------------+-----------------+------------------------+----------+
|kategori    |kota      |unit_terjual|harga_satuan|metode_pembayaran|estimasi_pengiriman_hari|dibatalkan|
+------------+----------+------------+------------+-----------------+------------------------+----------+
|Rumah Tangga|Solo      |14          |75000       |Transfer Bank    |14                      |1         |
|Makanan     |Solo      |2           |100000      |COD              |1                       |0         |
|Elektronik  |Yogyakarta|7           |75000       |Kartu Kredit     |14                      |1         |
|Elektronik  |Solo      |10          |150000      |COD              |9                       |1         |
|Makanan     |Solo      |4           |75000       |E-Wallet         |4                       |0         |
+------------+----------+------------+------------+-----------------+------------------------+----------+
only showing top 5 rows



C. Melatih Model dan Menyimpannya
-

Membagi Data Training dan Testing

In [6]:
# Membagi data dengan perbandingan 80:20
train, test = df.randomSplit([0.8, 0.2], seed=99)

print("Jumlah data training:", train.count())
print("Jumlah data testing:", test.count())

Jumlah data training: 766
Jumlah data testing: 234


Membuat Tahapan ML Pipeline

In [7]:
# Mengubah data kategorikal menjadi indeks angka
idx_kategori = StringIndexer(
    inputCol="kategori",
    outputCol="kategori_idx"
)

idx_kota = StringIndexer(
    inputCol="kota",
    outputCol="kota_idx"
)

idx_metode = StringIndexer(
    inputCol="metode_pembayaran",
    outputCol="metode_idx"
)

# Mengubah indeks menjadi representasi biner
ohe = OneHotEncoder(
    inputCols=["kategori_idx", "kota_idx", "metode_idx"],
    outputCols=["kategori_vec", "kota_vec", "metode_vec"]
)

# Menggabungkan seluruh fitur
assembler = VectorAssembler(
    inputCols=[
        "unit_terjual",
        "harga_satuan",
        "estimasi_pengiriman_hari",
        "kategori_vec",
        "kota_vec",
        "metode_vec"
    ],
    outputCol="features"
)

# Membuat model Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="dibatalkan",
    numTrees=25
)

# Menggabungkan seluruh tahapan ke dalam Pipeline
pipeline = Pipeline(
    stages=[
        idx_kategori,
        idx_kota,
        idx_metode,
        ohe,
        assembler,
        rf
    ]
)

print("ML Pipeline berhasil dibuat.")

ML Pipeline berhasil dibuat.


Melatih Model dan Membuat Prediksi

In [8]:
# Melatih model menggunakan data training
model = pipeline.fit(train)

print("Model Random Forest berhasil dilatih.")

# Membuat prediksi menggunakan data testing
predictions = model.transform(test)

print("Prediksi berhasil dibuat.")

# Menampilkan beberapa hasil prediksi
predictions.select(
    "kategori",
    "kota",
    "estimasi_pengiriman_hari",
    "dibatalkan",
    "prediction"
).show(10, truncate=False)

Model Random Forest berhasil dilatih.
Prediksi berhasil dibuat.
+----------+----------+------------------------+----------+----------+
|kategori  |kota      |estimasi_pengiriman_hari|dibatalkan|prediction|
+----------+----------+------------------------+----------+----------+
|Elektronik|Magelang  |7                       |1         |0.0       |
|Elektronik|Semarang  |13                      |1         |1.0       |
|Elektronik|Semarang  |12                      |1         |1.0       |
|Elektronik|Solo      |9                       |1         |1.0       |
|Elektronik|Solo      |13                      |1         |1.0       |
|Elektronik|Yogyakarta|14                      |1         |1.0       |
|Elektronik|Yogyakarta|14                      |1         |1.0       |
|Elektronik|Yogyakarta|5                       |0         |0.0       |
|Fashion   |Magelang  |11                      |1         |1.0       |
|Fashion   |Magelang  |14                      |1         |1.0       |
+----------+-

Evaluasi Model dan Menyimpan Model

In [9]:
# Membuat evaluator Accuracy
ev_acc = MulticlassClassificationEvaluator(
    labelCol="dibatalkan",
    predictionCol="prediction",
    metricName="accuracy"
)

# Membuat evaluator AUC-ROC
ev_auc = BinaryClassificationEvaluator(
    labelCol="dibatalkan",
    metricName="areaUnderROC"
)

# Menghitung hasil evaluasi
accuracy = ev_acc.evaluate(predictions)
auc = ev_auc.evaluate(predictions)

print("=== Hasil Evaluasi Model ===")
print(f"Accuracy : {accuracy:.4f}")
print(f"AUC-ROC  : {auc:.4f}")

# Menyimpan model
path_model_pembatalan = "model_pembatalan_tersimpan"

model.write().overwrite().save(path_model_pembatalan)

print("\nModel berhasil disimpan:", path_model_pembatalan)

=== Hasil Evaluasi Model ===
Accuracy : 0.8120
AUC-ROC  : 0.8841



Model berhasil disimpan: model_pembatalan_tersimpan


D. Simulasi Minggu Berikutnya
-

Menghentikan dan Membuat SparkSession Baru

In [10]:
# Menghentikan SparkSession lama
spark.stop()

print("SparkSession lama berhasil dihentikan.")

# Membuat SparkSession baru
spark = SparkSession.builder \
    .appName("Tugas13-MingguBerikutnya") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession baru berhasil dibuat.")
print("Application ID:", spark.sparkContext.applicationId)

# Memuat kembali model yang telah disimpan
model_dimuat = PipelineModel.load(
    path_model_pembatalan
)

print("Model berhasil dimuat kembali.")

SparkSession lama berhasil dihentikan.
SparkSession baru berhasil dibuat.
Application ID: local-1795678589630


Model berhasil dimuat kembali.


Membuat dan Menampilkan 10 Transaksi Baru

In [11]:
# Membuat 10 data transaksi baru
data_transaksi_baru = [
    {"kategori": "Elektronik", "kota": "Magelang", "unit_terjual": 3,
     "harga_satuan": 150000, "metode_pembayaran": "Transfer Bank",
     "estimasi_pengiriman_hari": 3},

    {"kategori": "Fashion", "kota": "Yogyakarta", "unit_terjual": 5,
     "harga_satuan": 75000, "metode_pembayaran": "E-Wallet",
     "estimasi_pengiriman_hari": 10},

    {"kategori": "Makanan", "kota": "Semarang", "unit_terjual": 8,
     "harga_satuan": 25000, "metode_pembayaran": "COD",
     "estimasi_pengiriman_hari": 5},

    {"kategori": "Rumah Tangga", "kota": "Solo", "unit_terjual": 2,
     "harga_satuan": 250000, "metode_pembayaran": "Kartu Kredit",
     "estimasi_pengiriman_hari": 13},

    {"kategori": "Kesehatan", "kota": "Magelang", "unit_terjual": 4,
     "harga_satuan": 50000, "metode_pembayaran": "E-Wallet",
     "estimasi_pengiriman_hari": 2},

    {"kategori": "Elektronik", "kota": "Yogyakarta", "unit_terjual": 6,
     "harga_satuan": 100000, "metode_pembayaran": "COD",
     "estimasi_pengiriman_hari": 12},

    {"kategori": "Fashion", "kota": "Semarang", "unit_terjual": 7,
     "harga_satuan": 50000, "metode_pembayaran": "Transfer Bank",
     "estimasi_pengiriman_hari": 7},

    {"kategori": "Makanan", "kota": "Solo", "unit_terjual": 10,
     "harga_satuan": 25000, "metode_pembayaran": "Kartu Kredit",
     "estimasi_pengiriman_hari": 14},

    {"kategori": "Rumah Tangga", "kota": "Magelang", "unit_terjual": 3,
     "harga_satuan": 150000, "metode_pembayaran": "COD",
     "estimasi_pengiriman_hari": 4},

    {"kategori": "Kesehatan", "kota": "Yogyakarta", "unit_terjual": 5,
     "harga_satuan": 75000, "metode_pembayaran": "Transfer Bank",
     "estimasi_pengiriman_hari": 11}
]

# Mengubah data menjadi Spark DataFrame
df_transaksi_baru = spark.createDataFrame(
    data_transaksi_baru
)

print("Jumlah transaksi baru:", df_transaksi_baru.count())

df_transaksi_baru.show(truncate=False)

Jumlah transaksi baru: 10
+------------------------+------------+------------+----------+-----------------+------------+
|estimasi_pengiriman_hari|harga_satuan|kategori    |kota      |metode_pembayaran|unit_terjual|
+------------------------+------------+------------+----------+-----------------+------------+
|3                       |150000      |Elektronik  |Magelang  |Transfer Bank    |3           |
|10                      |75000       |Fashion     |Yogyakarta|E-Wallet         |5           |
|5                       |25000       |Makanan     |Semarang  |COD              |8           |
|13                      |250000      |Rumah Tangga|Solo      |Kartu Kredit     |2           |
|2                       |50000       |Kesehatan   |Magelang  |E-Wallet         |4           |
|12                      |100000      |Elektronik  |Yogyakarta|COD              |6           |
|7                       |50000       |Fashion     |Semarang  |Transfer Bank    |7           |
|14                     

Memprediksi dan Menampilkan Hasilnya

In [12]:
# Menerapkan model yang sudah dimuat
hasil_prediksi = model_dimuat.transform(
    df_transaksi_baru
)

# Mengubah nama kolom prediction agar lebih jelas
hasil_prediksi = hasil_prediksi.withColumnRenamed(
    "prediction",
    "prediksi_dibatalkan"
)

# Menampilkan hasil prediksi
hasil_prediksi.select(
    "kategori",
    "kota",
    "unit_terjual",
    "harga_satuan",
    "metode_pembayaran",
    "estimasi_pengiriman_hari",
    "prediksi_dibatalkan",
    "probability"
).show(10, truncate=False)

+------------+----------+------------+------------+-----------------+------------------------+-------------------+----------------------------------------+
|kategori    |kota      |unit_terjual|harga_satuan|metode_pembayaran|estimasi_pengiriman_hari|prediksi_dibatalkan|probability                             |
+------------+----------+------------+------------+-----------------+------------------------+-------------------+----------------------------------------+
|Elektronik  |Magelang  |3           |150000      |Transfer Bank    |3                       |0.0                |[0.7806136592140173,0.2193863407859827] |
|Fashion     |Yogyakarta|5           |75000       |E-Wallet         |10                      |1.0                |[0.1938784610793315,0.8061215389206685] |
|Makanan     |Semarang  |8           |25000       |COD              |5                       |0.0                |[0.7233086835968363,0.27669131640316363]|
|Rumah Tangga|Solo      |2           |250000      |Kartu Kredit 

E. Menyimpan Laporan Akhir sebagai Parquet
-

Menyimpan, Membaca, dan Memeriksa Hasil Parquet

In [13]:
# Menyimpan hasil prediksi ke Parquet
hasil_prediksi.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet("laporan_risiko_pembatalan")

print("Laporan berhasil disimpan dalam format Parquet.")

# Membaca kembali data Parquet
df_laporan = spark.read.parquet(
    "laporan_risiko_pembatalan"
)

print("\nStruktur data Parquet:")
df_laporan.printSchema()

# Menghitung jumlah data
print("Jumlah data yang tersimpan:", df_laporan.count())

# Menampilkan hasil laporan
df_laporan.select(
    "kategori",
    "kota",
    "unit_terjual",
    "harga_satuan",
    "estimasi_pengiriman_hari",
    "prediksi_dibatalkan"
).show(10, truncate=False)

Laporan berhasil disimpan dalam format Parquet.

Struktur data Parquet:
root
 |-- estimasi_pengiriman_hari: long (nullable = true)
 |-- harga_satuan: long (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- unit_terjual: long (nullable = true)
 |-- kategori_idx: double (nullable = true)
 |-- kota_idx: double (nullable = true)
 |-- metode_idx: double (nullable = true)
 |-- kategori_vec: vector (nullable = true)
 |-- kota_vec: vector (nullable = true)
 |-- metode_vec: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediksi_dibatalkan: double (nullable = true)
 |-- kategori: string (nullable = true)

Jumlah data yang tersimpan: 10
+------------+----------+------------+------------+------------------------+-------------------+
|kategori    |kota      |unit_terjual|harga_satuan|estimasi_pengiriman_hari|prediksi_dibatalkan|
+--

F. Kesimpulan
-

Berdasarkan praktikum yang telah dilakukan, sistem deteksi risiko pembatalan transaksi berhasil dibuat melalui integrasi MongoDB, Spark MLlib, dan Parquet. MongoDB digunakan untuk menyimpan 1.000 dokumen transaksi mentah, kemudian data diekstraksi ke Spark DataFrame untuk diproses. Selanjutnya, model klasifikasi Random Forest dibangun menggunakan ML Pipeline yang terdiri dari StringIndexer, OneHotEncoder, dan VectorAssembler. Model dilatih menggunakan data training dan diuji menggunakan data testing dengan evaluasi accuracy dan AUC-ROC. Setelah itu, model disimpan agar dapat digunakan kembali tanpa melakukan pelatihan dari awal.

Berdasarkan dataset simulasi, estimasi pengiriman memiliki hubungan dengan kemungkinan transaksi dibatalkan. Semakin lama estimasi pengiriman, semakin besar peluang transaksi memperoleh label pembatalan. Oleh karena itu, tim operasional disarankan untuk memperhatikan transaksi dengan estimasi pengiriman yang panjang. Beberapa langkah yang dapat dilakukan adalah mempercepat proses pengiriman, memperbaiki pengelolaan stok, dan memberikan informasi estimasi pengiriman yang jelas kepada pelanggan. Hasil prediksi dapat digunakan sebagai alat bantu untuk menentukan transaksi yang perlu diprioritaskan. Namun, model tetap perlu diuji menggunakan data transaksi nyata agar hasil analisis lebih sesuai dengan kondisi bisnis.

Menghentikan Koneksi dan SparkSession
-

In [14]:
client.close()
spark.stop()
print("Koneksi MongoDB & SparkSession ditutup.")

Koneksi MongoDB & SparkSession ditutup.
